In [1]:
import os
import sys
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer


# gpu = sys.argv[1]
gpu = 0
os.environ["CUDA_VISIBLE_DEVICES"] = str(gpu)

/home/taerim/miniconda3/envs/FindMyLab/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
DIR = "./"

cmd = f"mkdir -p {DIR}/Embeddings/MPnet"
os.system(cmd)

0

In [ ]:
# year = int(sys.argv[2])

year = 1980  # Set a specific year for example. Iterate over multiple years in the final project.
print(f"Processing year: {year}")

Processing year: 1980


In [ ]:
# n_papers_per_year = 10000000000  # Set a large number to include all papers.
n_papers_per_year = 1000  # Set a small number for example.

df_out = []

# Read paper data.
df_yr = pd.read_csv(f"{DIR}/DataCollection/pubmed_data_{year}.tsv", sep="\t")

# Randomly sample "n_papers_per_year" papers from the year if there are more than "n_papers_per_year" papers.
if df_yr.shape[0] > n_papers_per_year:
    df_yr = df_yr.sample(n=n_papers_per_year, random_state=42)

df_out.append(df_yr)
print(f">>>>> Reading papers (year {year}): {df_yr.shape[0]} papers")

df_out = pd.concat(df_out, axis=0, ignore_index=True)
df_out["text_input"] = df_out["title"] + df_out["abstract"]

print(df_out.shape)
df_out.head()

>>>>> Reading papers (year 1980): 1000 papers
(1000, 10)


,Date,Year,PMID,title,abstract,journal,corresponding_authors,corresponding_affiliations,corresponding_countries,text_input
0,1980/10/01,1980,7191431,"Galactorrhea, oligo/amenorrhea, and hyperprola...",We report three patients with craniopharyngiom...,The Journal of clinical endocrinology and meta...,NaN,NaN,NaN,"Galactorrhea, oligo/amenorrhea, and hyperprola..."
1,1980/05/31,1980,7393033,Use of prophylactic antibiotics in surgery.,A prospective survey of 352 surgical patients ...,The Medical journal of Australia,NaN,NaN,NaN,Use of prophylactic antibiotics in surgery.A p...
2,1980/12/01,1980,7444045,The diagnostic and prognostic significance of ...,The significance of coronary artery calcificat...,Radiology,NaN,NaN,NaN,The diagnostic and prognostic significance of ...
3,1980/01/01,1980,7446209,Gangrene localized to the lower limbs in diabe...,"Over a period of eight years, 247 unselected p...",Acta medica Scandinavica,NaN,NaN,NaN,Gangrene localized to the lower limbs in diabe...
4,1980/07/01,1980,6155399,Aggregated human gamma-globulin-induced prolif...,Murine splenic B lymphocytes are stimulated to...,"Journal of immunology (Baltimore, Md. : 1950)",NaN,NaN,NaN,Aggregated human gamma-globulin-induced prolif...


# Extract Embeddings

### all-mpnet-base-v2 (https://huggingface.co/sentence-transformers/all-mpnet-base-v2)

In [5]:
model_general = SentenceTransformer("all-mpnet-base-v2", device="cuda")

In [ ]:
batch_size = 10000

embeddings_general = []
# embeddings_pmBERT = []

for i in range(0, df_out.shape[0], batch_size):
    print(
        f">>>>> Processing batch: {int((i/batch_size)+1)} ({i} - {min(i + batch_size, df_out.shape[0])})/{df_out.shape[0]}"
    )
    # print(f">>>>> Extracting : {i} - {min(i + batch_size, df_out.shape[0])}")
    batch_texts = df_out["text_input"].iloc[i : i + batch_size].tolist()

    # Generate embeddings for the batch using both models
    print(f"-Extracting embeddings with MPnet...")
    embeddings_general_tmp = model_general.encode(
        batch_texts, show_progress_bar=True, device="cuda"
    )

    # Append the embeddings to the respective lists
    embeddings_general.append(embeddings_general_tmp)
    print("")

# Concatenate the embeddings for the entire year
embeddings_general = np.concatenate(embeddings_general, axis=0)


# Save the embeddings to disk
np.save(f"{DIR}/Embeddings/MPnet/MPnet_embeddings_{year}.npy", embeddings_general)
print(f">>>>> Embeddings saved for year {year}.")

>>>>> Processing batch: 1 (0 - 1000)/1000
-Extracting embeddings with MPnet...


Batches: 100%|██████████| 32/32 [00:04<00:00,  7.76it/s]


>>>>> Embeddings saved for year 1980.
